# FinBERT Sentiment

In [18]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time

# 1. Directory Setup
base_dir = r"D:\MS_Data_Science_Thesis\Data_Extraction"
raw_folder = os.path.join(base_dir, "Raw_Data_Folder")

def extract_google_news_cotton():
    print("--- Starting Google News Archive Scraper for Cotton Sentiment ---")
    
    # 2. Set the temporal window
    start_year = 2010
    end_year = 2025
    
    all_headlines = []
    
    # 3. Iterate month-by-month to bypass RSS display limits
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            # Calculate the start and end of the month
            start_date = f"{year}-{month:02d}-01"
            if month == 12:
                end_date = f"{year+1}-01-01"
            else:
                end_date = f"{year}-{month+1:02d}-01"
            
            # The search query: Targeting financial and agricultural stress
            query = f'("cotton futures" OR "cotton crop" OR "cotton prices") after:{start_date} before:{end_date}'
            
            # Google News RSS Endpoint
            url = f"https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en"
            
            # A standard User-Agent header so the server knows we are a standard browser request
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }
            
            try:
                print(f"Fetching headlines for {start_date} to {end_date}...")
                response = requests.get(url, headers=headers)
                
                if response.status_code == 200:
                    # Parse the XML RSS feed
                    soup = BeautifulSoup(response.content, "xml")
                    items = soup.find_all("item")
                    
                    for item in items:
                        title = item.title.text
                        pub_date = item.pubDate.text
                        
                        # Filter to guarantee the headline is relevant
                        if 'cotton' in title.lower():
                            all_headlines.append({
                                'Date': pd.to_datetime(pub_date).date(),
                                'Text': title
                            })
                            
                # Sleep for 2 seconds between requests to be polite to the server and prevent IP blocks
                time.sleep(2)
                
            except Exception as e:
                print(f"Error fetching {start_date}: {e}")
                
    # 4. Compile, Clean, and Export
    if len(all_headlines) > 0:
        df = pd.DataFrame(all_headlines)
        
        # Drop identical headlines and sort chronologically
        df = df.drop_duplicates(subset=['Text'])
        df = df.sort_values('Date')
        
        output_path = os.path.join(raw_folder, "cotton_sentiment_raw.csv")
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        
        print(f"\nSuccess! Extracted {len(df)} historical headlines.")
        print(f"Data exported to: {output_path}")
    else:
        print("Failed to extract headlines. Check network connection.")

if __name__ == "__main__":
    extract_google_news_cotton()

--- Starting Google News Archive Scraper for Cotton Sentiment ---
Fetching headlines for 2010-01-01 to 2010-02-01...
Fetching headlines for 2010-02-01 to 2010-03-01...
Fetching headlines for 2010-03-01 to 2010-04-01...
Fetching headlines for 2010-04-01 to 2010-05-01...
Fetching headlines for 2010-05-01 to 2010-06-01...
Fetching headlines for 2010-06-01 to 2010-07-01...
Fetching headlines for 2010-07-01 to 2010-08-01...
Fetching headlines for 2010-08-01 to 2010-09-01...
Fetching headlines for 2010-09-01 to 2010-10-01...
Fetching headlines for 2010-10-01 to 2010-11-01...
Fetching headlines for 2010-11-01 to 2010-12-01...
Fetching headlines for 2010-12-01 to 2011-01-01...
Fetching headlines for 2011-01-01 to 2011-02-01...
Fetching headlines for 2011-02-01 to 2011-03-01...
Fetching headlines for 2011-03-01 to 2011-04-01...
Fetching headlines for 2011-04-01 to 2011-05-01...
Fetching headlines for 2011-05-01 to 2011-06-01...
Fetching headlines for 2011-06-01 to 2011-07-01...
Fetching headlin

In [24]:
pip install transformers torch

   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ------------------ --------------------- 5.0/10.7 MB 37.7 MB/s eta 0:00:01
   ---------------------------------------- 10.7/10.7 MB 47.7 MB/s  0:00:00
   ---------------------------------------- 0.0/616.3 kB ? eta -:--:--
   ---------------------------------------- 616.3/616.3 kB 22.1 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 3.7/3.7 MB 72.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 77.7 MB/s  0:00:00

   ----- ---------------------------------- 1/8 [regex]
  Attempting uninstall: hf-xet
   ----- ---------------------------------- 1/8 [regex]
    Found existing installation: hf-xet 1.2.0
   ----- ---------------------------------- 1/8 [regex]
   ---------- ----------------------------- 2/8 [hf-xet]
    Uninstalling hf-xet-1.2.0:
   ----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [35]:
%pip install --upgrade "torch>=2.6.0"

Note: you may need to restart the kernel to use updated packages.


In [36]:
%pip install "transformers<4.48.0"

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import os
import torch
from transformers import pipeline

# 1. Directory Setup
base_dir = r"D:\MS_Data_Science_Thesis\Data_Extraction"
raw_folder = os.path.join(base_dir, "Raw_Data_Folder")
input_file = os.path.join(raw_folder, "cotton_sentiment_raw.csv")

def generate_financial_sentiment_scores():
    print("--- Starting DistilRoBERTa Financial Sentiment Pipeline ---")
    
    if not os.path.exists(input_file):
        print(f"Error: Could not find {input_file}")
        return

    # 2. Load the text data
    df = pd.read_csv(input_file)
    df['Date'] = pd.to_datetime(df['Date'])
    
    # 3. Initialize the Secure Safetensors Model
    device = 0 if torch.cuda.is_available() else -1
    print(f"Loading Secure Safetensors Model... (Using {'GPU' if device == 0 else 'CPU'})")
    
    model_name = "mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis"
    
    # THE FIX: Added framework="pt" to force PyTorch and ignore TensorFlow
    sentiment_analyzer = pipeline(
        "sentiment-analysis", 
        model=model_name, 
        device=device, 
        framework="pt"
    )
    
    # 4. Process the headlines
    print(f"Scoring {len(df)} headlines. This will be very fast...")
    
    pos_scores, neg_scores, neu_scores = [], [], []
    
    for text in df['Text']:
        try:
            # Score the text
            result = sentiment_analyzer(str(text)[:512], return_all_scores=True)[0] 
            
            pos = next(item['score'] for item in result if item['label'].lower() == 'positive')
            neg = next(item['score'] for item in result if item['label'].lower() == 'negative')
            neu = next(item['score'] for item in result if item['label'].lower() == 'neutral')
            
            pos_scores.append(pos)
            neg_scores.append(neg)
            neu_scores.append(neu)
        except Exception as e:
            pos_scores.append(0)
            neg_scores.append(0)
            neu_scores.append(1) 
            
    # Add scores to the dataframe
    df['Sentiment_Positive'] = pos_scores
    df['Sentiment_Negative'] = neg_scores
    df['Sentiment_Neutral'] = neu_scores
    
    # 5. Aggregate by Date
    daily_sentiment = df.groupby('Date')[['Sentiment_Positive', 'Sentiment_Negative', 'Sentiment_Neutral']].mean().reset_index()
    daily_sentiment['Net_Sentiment'] = daily_sentiment['Sentiment_Positive'] - daily_sentiment['Sentiment_Negative']
    daily_sentiment = daily_sentiment.sort_values('Date')
    
    # 6. Export the scored data
    output_path = os.path.join(raw_folder, "cotton_sentiment_daily_scores.csv")
    daily_sentiment.to_csv(output_path, index=False)
    
    print(f"\nSuccess! Daily sentiment scores exported to: {output_path}")
    print(daily_sentiment.head())

if __name__ == "__main__":
    generate_financial_sentiment_scores()

--- Starting DistilRoBERTa Financial Sentiment Pipeline ---
Loading Secure Safetensors Model... (Using CPU)
Scoring 1082 headlines. This will be very fast...


C:\Users\siddh\anaconda3\envs\sidd_ds\Lib\site-packages\transformers\pipelines\text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(



Success! Daily sentiment scores exported to: D:\MS_Data_Science_Thesis\Data_Extraction\Raw_Data_Folder\cotton_sentiment_daily_scores.csv
        Date  Sentiment_Positive  Sentiment_Negative  Sentiment_Neutral  \
0 2010-04-07            0.000046            0.000377           0.999577   
1 2010-04-26            0.000605            0.989487           0.009908   
2 2010-06-20            0.000110            0.000188           0.999702   
3 2010-10-15            0.001023            0.997633           0.001343   
4 2010-10-27            0.492068            0.497383           0.010548   

   Net_Sentiment  
0      -0.000330  
1      -0.988882  
2      -0.000079  
3      -0.996610  
4      -0.005315  
